# Swiss Legal Retrieval — Colab GPU Pipeline

**執行順序**: Cell 1 → 2 → 3 → 4 → 5 → 6 → 7

| Cell | 功能 |
|------|------|
| 1 | Clone repo + 安裝套件 |
| 2 | 掛載 Google Drive，設定資料路徑 |
| 3 | 檢查 GPU |
| 4 | Kaggle API + 下載資料 |
| 5 | 設定 ANTHROPIC_API_KEY |
| 6 | 建立 BM25 + Dense (FAISS) 索引，存到 Drive |
| 7 | 執行 Hybrid Retrieval + 評估 macro F1 |

In [ ]:
# ── Cell 1: Clone repo + 安裝套件 ────────────────────────────────────────────
import os, sys

GITHUB_REPO = "https://github.com/gogovamosrafa/swiss-legal-retrieval.git"
REPO_DIR    = "/content/swiss-legal-retrieval"

if not os.path.exists(REPO_DIR):
    !git clone {GITHUB_REPO} {REPO_DIR}
else:
    print(f"Repo already exists, pulling latest...")
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}
sys.path.insert(0, f"{REPO_DIR}/src")

!pip install -q \
    rank-bm25 \
    sentence-transformers \
    faiss-gpu-cu12 \
    anthropic \
    python-dotenv \
    pyyaml \
    pandas \
    kaggle

print("\nPackages installed.")

In [ ]:
# ── Cell 2: 掛載 Google Drive，設定路徑 ──────────────────────────────────────
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive", force_remount=False)

# ↓ 改成你在 Drive 裡存放資料的路徑
DRIVE_ROOT = Path("/content/drive/MyDrive/swiss-legal-retrieval")
DATA_DIR   = DRIVE_ROOT / "data"
INDEX_DIR  = DRIVE_ROOT / "indices"
OUTPUT_DIR = Path("/content/output")   # 中間結果放 /content（不佔 Drive 空間）

DATA_DIR.mkdir(parents=True, exist_ok=True)
INDEX_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 設定環境變數讓 pipeline 讀取正確路徑
import os
os.environ["DATA_DIR"]   = str(DATA_DIR)
os.environ["INDEX_DIR"]  = str(INDEX_DIR)
os.environ["OUTPUT_DIR"] = str(OUTPUT_DIR)

LAWS_CSV   = DATA_DIR / "laws_de.csv"
COURTS_CSV = DATA_DIR / "court_considerations.csv"
TRAIN_CSV  = DATA_DIR / "train.csv"
VAL_CSV    = DATA_DIR / "val.csv"
TEST_CSV   = DATA_DIR / "test.csv"

print(f"DATA_DIR:   {DATA_DIR}")
print(f"INDEX_DIR:  {INDEX_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")
print()
for p in [LAWS_CSV, COURTS_CSV, TRAIN_CSV, VAL_CSV, TEST_CSV]:
    status = f"{p.stat().st_size/1e6:.0f} MB" if p.exists() else "MISSING"
    print(f"  {p.name:35s} {status}")

In [ ]:
# ── Cell 3: 檢查 GPU ──────────────────────────────────────────────────────────
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name}  ({vram_gb:.1f} GB VRAM)")
    DEVICE = "cuda"
else:
    print("[WARN] No GPU — dense encoding will be very slow.")
    print("  Runtime > Change runtime type > GPU")
    DEVICE = "cpu"

import faiss
print(f"FAISS version: {faiss.__version__}")

In [ ]:
# ── Cell 4: Kaggle API + 下載資料 ────────────────────────────────────────────
# 先到 Colab 左側欄 🔑 新增兩個 secrets:
#   KAGGLE_USERNAME  ← 你的 Kaggle username
#   KAGGLE_KEY       ← Kaggle API key（kaggle.com → Settings → API → Create New Token）

import os, json
from pathlib import Path
from google.colab import userdata

KAGGLE_JSON = Path("/root/.kaggle/kaggle.json")
KAGGLE_JSON.parent.mkdir(exist_ok=True)

username = userdata.get("KAGGLE_USERNAME")
key      = userdata.get("KAGGLE_KEY")

if username and key:
    KAGGLE_JSON.write_text(json.dumps({"username": username, "key": key}))
    KAGGLE_JSON.chmod(0o600)
    print(f"Kaggle credentials written for user: {username}")
else:
    print("[WARN] KAGGLE_USERNAME / KAGGLE_KEY secrets not found.")
    print("  Add them in Colab Secrets (key icon on left sidebar).")

# 下載競賽資料（若尚未下載）
COMP = "llm-agentic-legal-information-retrieval"
if not (DATA_DIR / "laws_de.csv").exists():
    print(f"\nDownloading competition data to {DATA_DIR} ...")
    !kaggle competitions download -c {COMP} -p {DATA_DIR} --unzip
    print("Download complete.")
else:
    print("Data already present, skipping download.")

# 確認檔案
for p in [DATA_DIR / "laws_de.csv", DATA_DIR / "court_considerations.csv",
          DATA_DIR / "train.csv", DATA_DIR / "val.csv", DATA_DIR / "test.csv"]:
    status = f"{p.stat().st_size/1e6:.0f} MB" if p.exists() else "MISSING"
    print(f"  {p.name:35s} {status}")

In [ ]:
# ── Cell 5: ANTHROPIC_API_KEY ────────────────────────────────────────────────
# Colab 左側欄 🔑 → 新增 secret: ANTHROPIC_API_KEY = sk-ant-...

import os
from google.colab import userdata

try:
    api_key = userdata.get("ANTHROPIC_API_KEY")
    if api_key:
        os.environ["ANTHROPIC_API_KEY"] = api_key
        print(f"ANTHROPIC_API_KEY loaded ({api_key[:12]}...{api_key[-4:]})")
    else:
        print("[WARN] ANTHROPIC_API_KEY secret is empty — reranking will be skipped.")
except Exception as e:
    print(f"[WARN] Could not read secret: {e}")

In [ ]:
# ── Cell 6: 建立 BM25 + Dense Index（存到 Drive，下次跳過）────────────────────
import csv, json, sys
import numpy as np
import faiss
from pathlib import Path
from sentence_transformers import SentenceTransformer

sys.path.insert(0, f"{REPO_DIR}/src")
from omnilex.retrieval.bm25_index import BM25Index

BM25_LAWS_PATH   = INDEX_DIR / "bm25_laws.pkl"
FAISS_LAWS_PATH  = INDEX_DIR / "dense_laws.faiss"
META_LAWS_PATH   = INDEX_DIR / "dense_laws_meta.jsonl"

DENSE_MODEL  = "intfloat/multilingual-e5-large"
PASSAGE_PFX  = "passage: "
BATCH_SIZE   = 256   # GPU can handle 256; drop to 64 if OOM

# ── Load CSV corpus ───────────────────────────────────────────────────────────
def load_csv(path, max_rows=None):
    docs = []
    with open(path, newline="", encoding="utf-8") as f:
        for i, row in enumerate(csv.DictReader(f)):
            if max_rows and i >= max_rows:
                break
            docs.append(dict(row))
    return docs

print("Loading laws corpus...")
laws = load_csv(LAWS_CSV)
print(f"  {len(laws):,} articles")

# ── BM25 (laws only) ──────────────────────────────────────────────────────────
if BM25_LAWS_PATH.exists():
    print(f"\nBM25 index found, loading...")
    bm25_laws = BM25Index.load(str(BM25_LAWS_PATH))
    print(f"  {len(bm25_laws.documents):,} docs")
else:
    print(f"\nBuilding BM25 index for {len(laws):,} laws...")
    bm25_laws = BM25Index(text_field="text", citation_field="citation")
    bm25_laws.build(laws)
    bm25_laws.save(str(BM25_LAWS_PATH))
    print(f"  Saved -> {BM25_LAWS_PATH}  ({BM25_LAWS_PATH.stat().st_size/1e6:.0f} MB)")

# ── Dense index (laws only, GPU-accelerated) ──────────────────────────────────
if FAISS_LAWS_PATH.exists():
    print(f"\nDense index found, loading...")
    faiss_laws = faiss.read_index(str(FAISS_LAWS_PATH))
    laws_meta  = load_csv(str(META_LAWS_PATH)) if META_LAWS_PATH.suffix == ".csv" else \
                 [json.loads(l) for l in open(META_LAWS_PATH, encoding="utf-8")]
    print(f"  {faiss_laws.ntotal:,} vectors")
else:
    print(f"\nBuilding dense index (multilingual-e5-large on {DEVICE})...")
    model = SentenceTransformer(DENSE_MODEL, device=DEVICE)
    model.max_seq_length = 512

    texts = [PASSAGE_PFX + (d.get("text") or "") for d in laws]
    print(f"  Encoding {len(texts):,} passages (batch={BATCH_SIZE})...")
    embeddings = model.encode(
        texts,
        batch_size=BATCH_SIZE,
        normalize_embeddings=True,
        show_progress_bar=True,
    ).astype("float32")

    dim = embeddings.shape[1]
    faiss_laws = faiss.IndexFlatIP(dim)
    faiss_laws.add(embeddings)
    faiss.write_index(faiss_laws, str(FAISS_LAWS_PATH))
    print(f"  Saved FAISS ({faiss_laws.ntotal:,} vectors) -> {FAISS_LAWS_PATH}")

    with open(META_LAWS_PATH, "w", encoding="utf-8") as f:
        for d in laws:
            f.write(json.dumps(d, ensure_ascii=False) + "\n")
    print(f"  Saved metadata -> {META_LAWS_PATH}")
    laws_meta = laws

print("\nIndex ready.")

In [ ]:
# ── Cell 7: Hybrid Retrieval + Evaluate ──────────────────────────────────────
import csv, json
from collections import defaultdict
from pathlib import Path

SPLIT = "val"   # "val" (10 queries, fast) or "train" (more queries)
QUERIES_CSV   = VAL_CSV if SPLIT == "val" else TRAIN_CSV
CANDIDATES_OUT = OUTPUT_DIR / f"candidates_{SPLIT}.jsonl"

BM25_TOP_K   = 100
DENSE_TOP_K  = 100
RRF_K        = 60
QUERY_PFX    = "query: "

# ── Load queries ──────────────────────────────────────────────────────────────
queries = []
with open(QUERIES_CSV, newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        queries.append(row)
print(f"Loaded {len(queries)} queries from {QUERIES_CSV.name}")

# ── Load dense model for query encoding ──────────────────────────────────────
from sentence_transformers import SentenceTransformer
if "model" not in dir() or model is None:
    model = SentenceTransformer(DENSE_MODEL, device=DEVICE)
    model.max_seq_length = 512

# ── Reciprocal Rank Fusion ────────────────────────────────────────────────────
def rrf(ranked_lists, k=60):
    scores = defaultdict(float)
    for ranked in ranked_lists:
        for rank, cid in enumerate(ranked, 1):
            scores[cid] += 1.0 / (rank + k)
    return sorted(scores, key=scores.get, reverse=True)

# ── Retrieve ──────────────────────────────────────────────────────────────────
all_docs = {d["citation"]: d for d in laws_meta}

with open(CANDIDATES_OUT, "w", encoding="utf-8") as fout:
    for row in queries:
        qid   = row["query_id"]
        query = row["query"]

        # BM25
        bm25_results = bm25_laws.search(query, top_k=BM25_TOP_K, return_scores=True)
        bm25_ranked  = [d.pop("citation", d.get("citation","")) for d in
                        sorted(bm25_results, key=lambda x: x.get("_score", 0), reverse=True)]

        # Dense
        q_emb = model.encode([QUERY_PFX + query], normalize_embeddings=True).astype("float32")
        scores, idxs = faiss_laws.search(q_emb, DENSE_TOP_K)
        dense_ranked = [laws_meta[i]["citation"] for i in idxs[0] if i >= 0]

        # RRF fusion
        fused = rrf([bm25_ranked, dense_ranked], k=RRF_K)

        candidates = [{"citation": cid, "text": all_docs.get(cid, {}).get("text", "")}
                      for cid in fused]
        fout.write(json.dumps({"query_id": qid, "query": query,
                                "candidates": candidates}, ensure_ascii=False) + "\n")

print(f"Saved {len(queries)} records -> {CANDIDATES_OUT}")

# ── Quick Evaluation (val/train only — has gold_citations) ────────────────────
if "gold_citations" in queries[0]:
    from omnilex.citations.normalizer import CitationNormalizer
    from omnilex.evaluation.metrics import citation_f1

    normalizer = CitationNormalizer()
    sep = ";"

    def parse_normalize(raw, sep=";"):
        parts = [c.strip() for c in raw.split(sep) if c.strip()]
        return {normalizer.canonicalize(c) for c in parts if normalizer.canonicalize(c)}

    TOP_K_PRED = 20   # how many candidates to treat as predictions
    f1s = []
    print(f"\nPer-query F1 (top-{TOP_K_PRED} candidates):")
    with open(CANDIDATES_OUT, encoding="utf-8") as f:
        for line in f:
            rec    = json.loads(line)
            qid    = rec["query_id"]
            gold_r = next(r["gold_citations"] for r in queries if r["query_id"] == qid)
            pred_c = [c["citation"] for c in rec["candidates"][:TOP_K_PRED]]

            gold_n = parse_normalize(gold_r)
            pred_n = {normalizer.canonicalize(c) for c in pred_c if normalizer.canonicalize(c)}

            m = citation_f1(pred_n, gold_n)
            f1s.append(m["f1"])
            print(f"  {qid}  F1={m['f1']:.3f}  P={m['precision']:.3f}  R={m['recall']:.3f}  "
                  f"pred={len(pred_n)} gold={len(gold_n)}")

    print(f"\nMacro F1 = {sum(f1s)/len(f1s):.4f}  ({len(f1s)} queries)")